In [ ]:
import polars
%load_ext autoreload
%autoreload 2
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.axes as axes

import polars as pl
import adbc_driver_postgresql.dbapi as dbapi


def gen_heatmap(cards: int) -> axes.Axes:
    with dbapi.connect("host=blue port=5432 dbname=bingo user=tokeiya3") as conn:
        df = pl.read_database("""
                              SELECT rank, round, count
                              FROM rank_round
                              WHERE cards = $1
                                AND rank <= 55
                              ORDER BY rank, round
                              """, connection=conn, execute_options={'parameters': [cards]})

    hdf = df.pivot(index='round', on='rank', values='count', aggregate_function='sum').fill_null(0).sort('round').drop(
        '0')
    plt.figure(figsize=(20, 10), dpi=200)
    ax = sns.heatmap(hdf.to_pandas().set_index("round"))
    ax.invert_yaxis()

    ax.set_xlabel('Rank')
    ax.set_ylabel('Round')
    ax.set_title(f'{cards} players')
    return ax


from typing_extensions import cast
from matplotlib.figure import Figure


def proc():
    for i in [x * 10 for x in range(1, 21)]:
        cast(Figure, gen_heatmap(i).figure).savefig(f'../images/rank_round_{i}.png', format='png', bbox_inches=None)
        print(f'heatmap_{i}.png saved')
    from PIL import Image

    paths: list[str] = []

    for i in [x * 10 for x in range(1, 21)]:
        paths.append(f'../images/rank_round_{i}.png')

    images = [Image.open(path).convert('RGBA') for path in paths]

    images[0].save(
        '../images/rank_round.webp',
        save_all=True,
        append_images=images[1:],
        duration=500,
        loop=0,
        lossless=True
    )


proc()